# Predicting Student Test Scores 
## Score: 8.71872

In [7]:
import time
import hashlib
import numpy as np
import pandas as pd

import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression

In [8]:
train = pd.read_csv('playground-series-s6e1/train.csv')

test = pd.read_csv('playground-series-s6e1/test.csv')

test_ids = test['id'].to_numpy()

y = train['exam_score'].to_numpy(dtype=float)

X = train.drop(columns=['id', 'exam_score'])
X_test = test.drop(columns=['id'])

ord_maps = {
    'sleep_quality': {'poor': 0, 'average': 1, 'good': 2},
    'facility_rating': {'low': 0, 'medium': 1, 'high': 2},
    'exam_difficulty': {'easy': 0, 'moderate': 1, 'hard': 2}
}

for col, mp in ord_maps.items():
    if col in X.columns:
        X[f'{col}_ord'] = X[col].map(mp).astype('float32')
        X_test[f'{col}_ord'] = X_test[col].map(mp).astype('float32')

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
for c in cat_cols:
    X[c] = X[c].astype('category')
    X_test[c] = X_test[c].astype('category')

if 'study_hours' in X.columns and 'sleep_quality_ord' in X.columns:
    X['int_study_x_sleepq'] = X['study_hours'].astype(float) * X['sleep_quality_ord'].astype(float)
    X_test['int_study_x_sleepq'] = X_test['study_hours'].astype(float) * X_test['sleep_quality_ord'].astype(float)

if 'study_hours' in X.columns and 'class_attendance' in X.columns:
    X['int_study_x_att'] = X['study_hours'].astype(float) * X['class_attendance'].astype(float)
    X_test['int_study_x_att'] = X_test['study_hours'].astype(float) * X_test['class_attendance'].astype(float)

if 'study_hours' in X.columns and 'sleep_hours' in X.columns:
    X['int_study_x_sleep'] = X['study_hours'].astype(float) * X['sleep_hours'].astype(float)
    X_test['int_study_x_sleep'] = X_test['study_hours'].astype(float) * X_test['sleep_hours'].astype(float)

if 'sleep_hours' in X.columns:
    X['sleep_opt_dist2_8'] = (X['sleep_hours'].astype(float) - 8.0) ** 2
    X_test['sleep_opt_dist2_8'] = (X_test['sleep_hours'].astype(float) - 8.0) ** 2

for c in ['study_hours', 'sleep_hours']:
    if c in X.columns:
        X[f'log1p_{c}'] = np.log1p(X[c].astype(float))
        X_test[f'log1p_{c}'] = np.log1p(X_test[c].astype(float))

bin_src_cols = [c for c in ['study_hours', 'sleep_hours', 'class_attendance'] if c in X.columns]
for c in bin_src_cols:
    _, bins = pd.qcut(X[c], q=20, duplicates='drop', retbins=True)
    bins[0] = -np.inf
    bins[-1] = np.inf
    bc = f'bin_{c}'
    X[bc] = pd.cut(X[c], bins=bins, include_lowest=True).cat.codes.astype('int16')
    X_test[bc] = pd.cut(X_test[c], bins=bins, include_lowest=True).cat.codes.astype('int16')


In [9]:
base_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': 0.03,
    'n_estimators': 8000,
    'num_leaves': 79,
    'max_depth': 10,
    'min_child_samples': 55,
    'reg_alpha': 10.0,
    'reg_lambda': 0.50,
    'min_split_gain': 1e-6,
    'subsample': 0.72,
    'subsample_freq': 3,
    'colsample_bytree': 0.65,
    'n_jobs': -1,
    'force_col_wise': True
}

seeds = [420, 666, 80085]
n_splits = 10

EARLY_STOP = 250
MAX_SECONDS = 3300

USE_CATBOOST = False

FAST_TUNE = False
FAST_TUNE_SEED = 420
FAST_TUNE_SPLITS = 5
FAST_TUNE_SMOOTHS = [5.0, 10.0, 25.0, 50.0]
FAST_TUNE_REG_ALPHA = [5.0, 10.0, 20.0]

TE_SMOOTH = 5.0
BEST_REG_ALPHA = 10.0
base_params['reg_alpha'] = BEST_REG_ALPHA

te_cols = [c for c in ['course', 'exam_difficulty', 'study_method', 'sleep_quality', 'facility_rating', 'internet_access', 'gender'] if c in X.columns]
te_cols += [c for c in X.columns if str(c).startswith('bin_')]
te_cols = list(dict.fromkeys(te_cols))

te_pairs = []
for a, b in [('course', 'exam_difficulty'), ('study_method', 'exam_difficulty'), ('course', 'study_method')]:
    if a in X.columns and b in X.columns:
        te_pairs.append((a, b))

for c in te_cols:
    vc = pd.concat([X[c], X_test[c]]).value_counts(dropna=False)
    X[f'ce_{c}'] = X[c].map(vc).astype(float).fillna(0.0)
    X_test[f'ce_{c}'] = X_test[c].map(vc).astype(float).fillna(0.0)

t0 = time.time()

alt_params = {
    **base_params,
    'num_leaves': 47,
    'max_depth': -1,
    'min_child_samples': 90,
    'reg_alpha': 20.0,
    'reg_lambda': 1.50,
    'subsample': 0.85,
    'subsample_freq': 1,
    'colsample_bytree': 0.85
}

alt_params['reg_alpha'] = BEST_REG_ALPHA * 2.0

dart_params = {
    **base_params,
    'boosting_type': 'dart',
    'n_estimators': 2500,
    'drop_rate': 0.10,
    'skip_drop': 0.50,
    'num_leaves': 127,
    'max_depth': -1,
    'min_child_samples': 30,
    'subsample': 0.80,
    'subsample_freq': 1,
    'colsample_bytree': 0.80,
    'reg_alpha': 2.0,
    'reg_lambda': 1.0
}

extra_trees_params = {
    **base_params,
    'extra_trees': True,
    'num_leaves': 127,
    'max_depth': -1,
    'min_child_samples': 30,
    'subsample': 0.90,
    'subsample_freq': 1,
    'colsample_bytree': 0.90,
    'reg_alpha': 2.0,
    'reg_lambda': 1.0
}


y_bins_cat = pd.qcut(pd.Series(y), q=20, duplicates='drop')
y_bins = y_bins_cat.cat.codes.to_numpy()
min_count = int(pd.Series(y_bins).value_counts().min())
if n_splits > min_count:
    n_splits = max(2, min_count)
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

def te_fit_1_stats(X_ref, y_ref, col):
    y_s = pd.Series(y_ref, index=X_ref.index).astype(float)
    g = y_s.groupby(X_ref[col], observed=False).agg(['sum', 'count'])
    prior = float(y_s.mean())
    return g['sum'], g['count'], prior

def te_fit_2_stats(X_ref, y_ref, a, b):
    y_s = pd.Series(y_ref, index=X_ref.index).astype(float)
    key = X_ref[a].astype(str) + '|' + X_ref[b].astype(str)
    g = y_s.groupby(key).agg(['sum', 'count'])
    prior = float(y_s.mean())
    return g['sum'], g['count'], prior

def te_apply_fold_1(X_df, col, sum_s, cnt_s, prior, smooth):
    s = X_df[col].map(sum_s)
    c = X_df[col].map(cnt_s)
    s = s.astype(float)
    c = c.astype(float)
    enc = (s + prior * smooth) / (c + smooth)
    return enc.fillna(prior)

def te_apply_fold_2(X_df, a, b, sum_s, cnt_s, prior, smooth):
    key = X_df[a].astype(str) + '|' + X_df[b].astype(str)
    s = key.map(sum_s)
    c = key.map(cnt_s)
    s = s.astype(float)
    c = c.astype(float)
    enc = (s + prior * smooth) / (c + smooth)
    return enc.fillna(prior)

def te_apply_loo_1(X_df, y_ref, col, sum_s, cnt_s, prior, smooth):
    y_s = pd.Series(y_ref, index=X_df.index).astype(float)
    s = X_df[col].map(sum_s).astype(float)
    c = X_df[col].map(cnt_s).astype(float)
    s2 = s - y_s
    c2 = c - 1.0
    enc = (s2 + prior * smooth) / (c2 + smooth)
    enc = enc.where(c2 > 0.0, prior)
    return enc.fillna(prior)

def te_apply_loo_2(X_df, y_ref, a, b, sum_s, cnt_s, prior, smooth):
    y_s = pd.Series(y_ref, index=X_df.index).astype(float)
    key = X_df[a].astype(str) + '|' + X_df[b].astype(str)
    s = key.map(sum_s).astype(float)
    c = key.map(cnt_s).astype(float)
    s2 = s - y_s
    c2 = c - 1.0
    enc = (s2 + prior * smooth) / (c2 + smooth)
    enc = enc.where(c2 > 0.0, prior)
    return enc.fillna(prior)

def add_te(X_tr, X_va, X_te, y_tr):
    add_tr = {}
    add_va = {}
    add_te2 = {}

    for c in te_cols:
        sum_s, cnt_s, prior = te_fit_1_stats(X_tr, y_tr, c)
        name = f'te_{c}'
        add_tr[name] = te_apply_fold_1(X_tr, c, sum_s, cnt_s, prior, TE_SMOOTH)
        add_va[name] = te_apply_fold_1(X_va, c, sum_s, cnt_s, prior, TE_SMOOTH)
        add_te2[name] = te_apply_fold_1(X_te, c, sum_s, cnt_s, prior, TE_SMOOTH)

    for a, b in te_pairs:
        sum_s, cnt_s, prior = te_fit_2_stats(X_tr, y_tr, a, b)
        name = f'te_{a}__{b}'
        add_tr[name] = te_apply_fold_2(X_tr, a, b, sum_s, cnt_s, prior, TE_SMOOTH)
        add_va[name] = te_apply_fold_2(X_va, a, b, sum_s, cnt_s, prior, TE_SMOOTH)
        add_te2[name] = te_apply_fold_2(X_te, a, b, sum_s, cnt_s, prior, TE_SMOOTH)

    X_tr2 = pd.concat([X_tr, pd.DataFrame(add_tr, index=X_tr.index)], axis=1)
    X_va2 = pd.concat([X_va, pd.DataFrame(add_va, index=X_va.index)], axis=1)
    X_te2 = pd.concat([X_te, pd.DataFrame(add_te2, index=X_te.index)], axis=1)

    return X_tr2, X_va2, X_te2

if FAST_TUNE:
    ft_bins_cat = pd.qcut(pd.Series(y), q=20, duplicates='drop')
    ft_bins = ft_bins_cat.cat.codes.to_numpy()
    ft_min_count = int(pd.Series(ft_bins).value_counts().min())
    ft_splits = FAST_TUNE_SPLITS
    if ft_splits > ft_min_count:
        ft_splits = max(2, ft_min_count)
    ft_skf = StratifiedKFold(n_splits=ft_splits, shuffle=True, random_state=123)

    best = (float('inf'), None, None)

    for sm in FAST_TUNE_SMOOTHS:
        for ra in FAST_TUNE_REG_ALPHA:
            TE_SMOOTH = float(sm)
            base_params['reg_alpha'] = float(ra)
            alt_params['reg_alpha'] = float(ra) * 2.0

            oof = np.full(len(X), np.nan, dtype=float)

            for fold, (tr_idx, va_idx) in enumerate(ft_skf.split(X, ft_bins), start=1):
                X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
                y_tr, y_va = y[tr_idx], y[va_idx]

                X_tr2, X_va2, _ = add_te(X_tr, X_va, X_va, y_tr)

                p = {**base_params, 'random_state': FAST_TUNE_SEED}
                model = lgb.LGBMRegressor(**p)
                model.fit(
                    X_tr2,
                    y_tr,
                    eval_set=[(X_va2, y_va)],
                    callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)]
                )

                oof[va_idx] = model.predict(X_va2)

            filled = ~np.isnan(oof)
            rmse = float(np.sqrt(mean_squared_error(y[filled], np.clip(oof[filled], 0, 100))))

            print('FAST_TUNE', 'TE_SMOOTH', TE_SMOOTH, 'reg_alpha', base_params['reg_alpha'], 'rmse', rmse)

            if rmse < best[0]:
                best = (rmse, TE_SMOOTH, base_params['reg_alpha'])

    TE_SMOOTH = float(best[1])
    base_params['reg_alpha'] = float(best[2])
    alt_params['reg_alpha'] = float(best[2]) * 2.0
    print('FAST_TUNE BEST', 'rmse', best[0], 'TE_SMOOTH', TE_SMOOTH, 'reg_alpha', base_params['reg_alpha'])

def cv_run(params_list, seeds, label):
    sum_oof = np.zeros(len(X), dtype=float)
    cnt_oof = np.zeros(len(X), dtype=float)
    sum_test = np.zeros(len(X_test), dtype=float)
    seeds_done = 0

    for s_i, seed in enumerate(seeds, start=1):
        params_list2 = params_list if isinstance(params_list, (list, tuple)) else [params_list]

        oof = np.full(len(X), np.nan, dtype=float)
        test_pred_sum = np.zeros(len(X_test), dtype=float)
        rmse_scores = []
        folds_done = 0

        print(f'{label} SEED {seed} ({s_i}/{len(seeds)})')

        for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y_bins), start=1):
            if (time.time() - t0) > MAX_SECONDS:
                break

            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y[tr_idx], y[va_idx]

            X_tr2, X_va2, X_test2 = add_te(X_tr, X_va, X_test, y_tr)

            va_preds = []
            te_preds = []
            rmses = []

            for params in params_list2:
                p = {**params, 'random_state': seed}
                model = lgb.LGBMRegressor(**p)

                bt = str(p.get('boosting_type', 'gbdt'))
                if bt == 'dart':
                    cb = [lgb.log_evaluation(200)]
                else:
                    cb = [lgb.early_stopping(EARLY_STOP), lgb.log_evaluation(200)]

                print('    variant', bt, 'start')

                model.fit(
                    X_tr2,
                    y_tr,
                    eval_set=[(X_va2, y_va)],
                    callbacks=cb
                )

                va_p = model.predict(X_va2)
                te_p = model.predict(X_test2)
                rmse_p = float(np.sqrt(mean_squared_error(y_va, va_p)))

                va_preds.append(va_p)
                te_preds.append(te_p)
                rmses.append(rmse_p)

            inv = 1.0 / (np.square(np.array(rmses, dtype=float)) + 1e-12)
            w = inv / inv.sum()

            va_pred = np.zeros(len(va_idx), dtype=float)
            te_pred = np.zeros(len(X_test), dtype=float)
            for wi, va_p, te_p in zip(w, va_preds, te_preds):
                va_pred += wi * va_p
                te_pred += wi * te_p

            oof[va_idx] = va_pred

            fold_rmse = float(np.sqrt(mean_squared_error(y_va, va_pred)))
            rmse_scores.append(fold_rmse)
            print(f'  Fold {fold}/{n_splits} RMSE: {fold_rmse:.5f} | w: {w.tolist()} | rmse: {rmses}')

            test_pred_sum += te_pred
            folds_done += 1

        if folds_done == 0:
            break

        test_pred = test_pred_sum / folds_done

        filled = ~np.isnan(oof)
        oof_filled = np.clip(oof[filled], 0, 100)
        sum_oof[filled] += oof_filled
        cnt_oof[filled] += 1.0

        sum_test += np.clip(test_pred, 0, 100)
        seeds_done += 1

        oof_rmse = float(np.sqrt(mean_squared_error(y[filled], oof_filled)))
        print(f'{label} Seed {seed} OOF RMSE: {oof_rmse:.5f} | Mean fold: {np.mean(rmse_scores):.5f} (+/- {np.std(rmse_scores):.5f})')

        if (time.time() - t0) > MAX_SECONDS:
            break

    denom = np.maximum(cnt_oof, 1.0)
    all_oof = np.clip(sum_oof / denom, 0, 100)

    if seeds_done > 0:
        all_test = np.clip(sum_test / seeds_done, 0, 100)
    else:
        all_test = np.zeros(len(X_test), dtype=float)

    filled_all = cnt_oof > 0
    if filled_all.any():
        final_oof_rmse = float(np.sqrt(mean_squared_error(y[filled_all], all_oof[filled_all])))
    else:
        final_oof_rmse = float('nan')

    print(f'{label} FINAL OOF RMSE: {final_oof_rmse:.5f}')

    return all_oof, all_test, final_oof_rmse


def cv_run_cb(params, seed, label, n_splits_cb=3):
    skf_cb = StratifiedKFold(n_splits=n_splits_cb, shuffle=True, random_state=seed)

    oof = np.full(len(X), np.nan, dtype=float)
    test_pred_sum = np.zeros(len(X_test), dtype=float)
    rmse_scores = []
    folds_done = 0

    cb_cat_cols = [c for c in te_cols if c in X.columns]

    print(f'{label} SEED {seed} (1/1)')

    for fold, (tr_idx, va_idx) in enumerate(skf_cb.split(X, y_bins), start=1):
        if (time.time() - t0) > MAX_SECONDS:
            break

        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        X_tr2, X_va2, X_test2 = add_te(X_tr, X_va, X_test, y_tr)

        cat_features = [int(X_tr2.columns.get_loc(c)) for c in cb_cat_cols if c in X_tr2.columns]

        tr_pool = Pool(X_tr2, y_tr, cat_features=cat_features)
        va_pool = Pool(X_va2, y_va, cat_features=cat_features)
        te_pool = Pool(X_test2, cat_features=cat_features)

        model = CatBoostRegressor(
            **params,
            random_seed=seed,
            allow_writing_files=False
        )

        print(f'  Fold {fold}/{n_splits_cb} start')

        model.fit(tr_pool, eval_set=va_pool, use_best_model=True, verbose=200)

        va_pred = model.predict(va_pool)
        te_pred = model.predict(te_pool)

        oof[va_idx] = va_pred
        test_pred_sum += te_pred
        folds_done += 1

        fold_rmse = float(np.sqrt(mean_squared_error(y_va, va_pred)))
        rmse_scores.append(fold_rmse)
        print(f'  Fold {fold}/{n_splits_cb} RMSE: {fold_rmse:.5f}')

    if folds_done == 0:
        all_test = np.full(len(X_test), np.nan, dtype=float)
    else:
        all_test = test_pred_sum / folds_done

    filled = ~np.isnan(oof)
    if filled.any():
        oof_rmse = float(np.sqrt(mean_squared_error(y[filled], np.clip(oof[filled], 0, 100))))
    else:
        oof_rmse = float('nan')

    print(f'{label} FINAL OOF RMSE: {oof_rmse:.5f} | Mean fold: {np.mean(rmse_scores) if rmse_scores else float("nan"):.5f} (+/- {np.std(rmse_scores) if rmse_scores else float("nan"):.5f})')

    return np.clip(oof, 0, 100), np.clip(all_test, 0, 100), oof_rmse


lgb_oof, lgb_test, _ = cv_run([base_params, alt_params, dart_params, extra_trees_params], seeds, 'LGB4')

if USE_CATBOOST:
    cb_params = {
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'iterations': 2500,
        'learning_rate': 0.03,
        'depth': 8,
        'l2_leaf_reg': 6.0,
        'random_strength': 1.0,
        'bagging_temperature': 0.5,
        'subsample': 0.80,
        'rsm': 0.85,
        'min_data_in_leaf': 25,
        'od_type': 'Iter',
        'od_wait': 100,
        'thread_count': -1
    }

    cb_oof, cb_test, _ = cv_run_cb(cb_params, seed=2025, label='CAT', n_splits_cb=3)

    cb_oof2 = np.array(cb_oof, dtype=float, copy=True)
    cb_test2 = np.array(cb_test, dtype=float, copy=True)

    m_oof = np.isnan(cb_oof2)
    if m_oof.any():
        cb_oof2[m_oof] = lgb_oof[m_oof]

    m_test = np.isnan(cb_test2)
    if m_test.any():
        cb_test2[m_test] = lgb_test[m_test]

    blend_X = np.vstack([lgb_oof, cb_oof2]).T
    blend_lr = LinearRegression(positive=True)
    blend_lr.fit(blend_X, y)

    blend_oof = blend_lr.predict(blend_X)
    blend_rmse = float(np.sqrt(mean_squared_error(y, blend_oof)))
    print('BLEND coef', blend_lr.coef_.tolist(), 'intercept', float(blend_lr.intercept_), 'rmse', blend_rmse)

    pred = blend_lr.predict(np.vstack([lgb_test, cb_test2]).T)
    pred = np.clip(pred, 0, 100)
else:
    pred = np.clip(lgb_test, 0, 100)

submission = pd.DataFrame({'id': test_ids, 'exam_score': pred})

out_path = 'submission.csv'
submission.to_csv(out_path, index=False)
with open(out_path, 'rb') as f:
    md5 = hashlib.md5(f.read()).hexdigest()

print(out_path)
print('md5', md5)
print('pred_mean', float(np.mean(pred)), 'pred_std', float(np.std(pred)), 'pred_min', float(np.min(pred)), 'pred_max', float(np.max(pred)))
print('pred_ge_99_5', int(np.sum(pred >= 99.5)), 'pred_ge_97', int(np.sum(pred >= 97.0)))


LGB4 SEED 420 (1/3)
    variant gbdt start
Training until validation scores don't improve for 250 rounds
[200]	valid_0's rmse: 8.79797
[400]	valid_0's rmse: 8.76661
[600]	valid_0's rmse: 8.75314
[800]	valid_0's rmse: 8.74455
[1000]	valid_0's rmse: 8.73978
[1200]	valid_0's rmse: 8.73733
[1400]	valid_0's rmse: 8.73673
[1600]	valid_0's rmse: 8.73541
[1800]	valid_0's rmse: 8.73613
Early stopping, best iteration is:
[1589]	valid_0's rmse: 8.73533
    variant gbdt start
Training until validation scores don't improve for 250 rounds
[200]	valid_0's rmse: 8.81881
[400]	valid_0's rmse: 8.78034
[600]	valid_0's rmse: 8.76566
[800]	valid_0's rmse: 8.75742
[1000]	valid_0's rmse: 8.75098
[1200]	valid_0's rmse: 8.74626
[1400]	valid_0's rmse: 8.74277
[1600]	valid_0's rmse: 8.74005
[1800]	valid_0's rmse: 8.73824
[2000]	valid_0's rmse: 8.73656
[2200]	valid_0's rmse: 8.73497
[2400]	valid_0's rmse: 8.73381
[2600]	valid_0's rmse: 8.7329
[2800]	valid_0's rmse: 8.73233
[3000]	valid_0's rmse: 8.73226
[3200]	va